In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import LabelEncoder

In [31]:
# ==========================
# Project Paths
# ==========================

RAW_DATA = "/Users/mariam/Downloads/New_Master copy/smartshop/offline_guideline_pipeline/data/raw/data.csv"

PROCESSED_FOLDER = Path("data/processed")
REPORT_FOLDER = Path("reports")

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)
REPORT_FOLDER.mkdir(parents=True, exist_ok=True)

Step 2 — Load Dataset

In [32]:
df = pd.read_csv(RAW_DATA)

print("="*50)
print("Dataset Loaded Successfully")
print("="*50)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Dataset Loaded Successfully
Rows: 235
Columns: 74


In [34]:
quality_report = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

quality_report.to_excel(
    PROCESSED_FOLDER / "cleaning_report.xlsx",
    index=False
)

In [35]:
original_rows = len(df)
original_rows

235

In [36]:

# check1,2,4,5 should be 3; check3 should be 'Rounded Corners'
df_clean = df[
    (df['check1'] == 3) &
    (df['check2'] == 3) &
    (df['check3'] == 'Rounded Corners') &
    (df['check4'] == 3) &
    (df['check5'] == 3)
].copy()

print(f'After attention checks: {df_clean.shape}')
print(f'Removed: {len(df) - len(df_clean)} rows')

After attention checks: (195, 74)
Removed: 40 rows


In [37]:
df_clean = df_clean.dropna(how="all")

In [38]:
df=df_clean

In [39]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )



In [41]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing.to_csv(
    REPORT_FOLDER / "missing_values.csv"
)

In [42]:
df.to_csv(
    PROCESSED_FOLDER / "clean_dataset.csv",
    index=False
)

In [43]:
encoded_df = df.copy()

encoders = {}

In [44]:
for col in encoded_df.columns:

    if encoded_df[col].dtype == object:

        le = LabelEncoder()

        encoded_df[col] = le.fit_transform(
            encoded_df[col].astype(str)
        )

        encoders[col] = le

In [45]:
encoded_df.to_csv(
    PROCESSED_FOLDER / "encoded_dataset.csv",
    index=False
)

In [47]:
summary = f"""
=========================
Cleaning Summary
=========================

Original participants : {original_rows}

Final participants    : {len(df)}

Columns : {df.shape[1]}

Clean dataset saved.

Encoded dataset saved.

=========================
"""

In [48]:
with open(
    REPORT_FOLDER / "cleaning_summary.txt",
    "w"
) as file:

    file.write(summary)

print(summary)



Cleaning Summary

Original participants : 235

Final participants    : 195

Columns : 74

Clean dataset saved.

Encoded dataset saved.




In [49]:
import joblib

joblib.dump(encoders, "data/processed/label_encoders.pkl")

['data/processed/label_encoders.pkl']

Phase 2